# Assignment 01

In [12]:
! pip install kaggle
import pandas as pd
import os


^C
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/build_env.py", line 19, in <module>
    from pip._internal.cli.spinners import open_spinner
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/spinners.py", line 9, in <module>
    from pip._internal.utils.logging import get_indentation
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/u

| Requirement | Details |
|--------------|----------|
| Name of the Dataset | Web Traffic Time Series Forecasting |
| Source (Direct Link) | Kaggle Competition Page |
| Data Description | The dataset measures the number of daily views/traffic for approximately 145,000 Wikipedia articles. Each time series corresponds to a specific article, language, access type (all, mobile, desktop), and agent (e.g., spider). |
| Frequency of the Data | Daily |
| Time Span Covered | The primary training data spans from July 1, 2015 to September 10, 2017. (The first stage of the competition used data up to December 31, 2016, with the full dataset extending to September 2017). |


In [13]:

def load_and_merge_data(stage=1):

    if stage not in [1, 2]:
        raise ValueError("Stage must be 1 or 2.")

    train_file = f'/kaggle/input/web-traffic-time-series-forecasting/train_{stage}.csv.zip'
    key_file = f'/kaggle/input/web-traffic-time-series-forecasting/key_{stage}.csv.zip'


    print(f"Loading data for Stage {stage}...")
    try:
        print(f"Loading {train_file} (Web Traffic Views)...")
        train_df = pd.read_csv(train_file)

        print(f"Loading {key_file} (Page ID mapping)...")
        key_df = pd.read_csv(key_file)

        print(f"Train data shape: {train_df.shape}")
        print(f"Key data shape: {key_df.shape}")

    except FileNotFoundError as e:
        print(f"Error: {e}. Please ensure the zipped files ({train_file}, {key_file}) are present in the Kaggle Notebook's input directory.")
        raise


    print("Melting the wide-format training data...")
    date_cols = train_df.columns[1:] 

    train_long_df = pd.melt(
        train_df, 
        id_vars=['Page'], 
        value_vars=date_cols, 
        var_name='Date', 
        value_name='Visits'
    )


    train_long_df['Date'] = pd.to_datetime(train_long_df['Date'])
    train_long_df['Visits'] = pd.to_numeric(train_long_df['Visits']) 


    print("Merging training data with key data...")
    merged_df = pd.merge(train_long_df, key_df, on='Page', how='left')

    print("\n--- Final Merged, Long-Format DataFrame Head ---")
    print(merged_df.head())

    print("\n--- DataFrame Information ---")
    merged_df.info()
    
    return merged_df




In [14]:
def check_missing_values(df):

    print("\n--- Missing Value Analysis ---")
    if df.isnull().values.any():
        print("MISSING DATA FOUND: Yes, the DataFrame contains NaN values.")
        
        nan_counts = df.isnull().sum()
        print("\nNaN Count per Column:")
        print(nan_counts[nan_counts > 0].sort_values(ascending=False))
        
    else:
        print("MISSING DATA FOUND: No, the DataFrame is complete (no NaN values).")

In [15]:
def check_duplicates(df):

    print("\n--- Duplicate Value Analysis ---")

    total_duplicates = df.duplicated().sum()
    print(f"Total number of exact duplicate rows (across all columns): {total_duplicates}")
    
    key_cols = ['Page', 'Date', 'Id']
    key_duplicates = df.duplicated(subset=key_cols).sum()
    
    if key_duplicates > 0:
        print(f"Total number of duplicate time series entries (Page + Date + Id): {key_duplicates}")
        print("ACTION REQUIRED: This indicates that a page has more than one recorded visit count for the same day.")

        sample_duplicates = df[df.duplicated(subset=key_cols, keep=False)].sort_values(by=['Page', 'Date']).head(10)
        print("\nSample (Head) of Duplicate Time Series Rows (showing both copies):")
        print(sample_duplicates)
    else:
        print("SUCCESS: No duplicate time series entries found (unique Page and Date combinations).")

In [17]:
def validate_timestamp_sequence(df):

    unique_dates = df['Date'].sort_values().unique()
    
    if len(unique_dates) > 1:
        date_differences = pd.Series(unique_dates).diff().dt.days
        date_gaps = (date_differences > 1).sum()
    else:
        date_gaps = 0

    if date_gaps == 0:
        print("GLOBAL CHECK SUCCESS: The full dataset's date range is continuous (no missing days in the index).")
    else:
        print(f"GLOBAL CHECK WARNING: Found {date_gaps} gap(s) of 2 or more days in the overall date sequence.")


    actual_set = set(df[['Page', 'Date']].itertuples(index=False))
    
    expected_pages = df['Page'].unique()
    expected_dates = unique_dates
    
    expected_set = set(product(expected_pages, expected_dates))

    missing_entries = expected_set - actual_set
    
    if len(missing_entries) == 0:
        print("INTERNAL CHECK SUCCESS: Every Page has an entry (row) for every Date in the global range.")
        print("Note: The 'Visits' value in these rows may still be NaN (checked by check_missing_values).")
    else:
        print(f"INTERNAL CHECK CRITICAL: Found {len(missing_entries)} missing Page-Date combinations (structural data loss).")
        print("Sample of structurally missing entries (Page, Date):")
        print(list(missing_entries)[:5])
        print("This suggests an issue during the melt or merge process.")

### Loading data

In [ ]:
try:
    print("--- Starting Full Data Loading and Concatenation Process ---")
    stage_1_data = load_and_merge_data(stage=1)
    
    print("\n" + "="*60)
    print("Starting Stage 2 Data Load...")
    stage_2_data = load_and_merge_data(stage=2)
    print("="*60)

    print("\nConcatenating Stage 1 and Stage 2 data...")
    final_merged_data = pd.concat([stage_1_data, stage_2_data], ignore_index=True)
    
    print("\n--- Final Concatenated (All) DataFrame Summary ---")
    print(f"Shape of Stage 1 data: {stage_1_data.shape}")
    print(f"Shape of Stage 2 data: {stage_2_data.shape}")
    print(f"Total rows in combined dataset (all_data): {all_data.shape[0]}")
    print(final_merged_data.head())
   
    stage_1_data = []
    stage_2_data = []
    
    print("\nData loading and merging complete.")
    
except Exception as e:
    print(f"\nFailed to process data: {e}")




--- Starting Full Data Loading and Concatenation Process ---
Loading data for Stage 1...
Loading /kaggle/input/web-traffic-time-series-forecasting/train_1.csv.zip (Web Traffic Views)...
Loading /kaggle/input/web-traffic-time-series-forecasting/key_1.csv.zip (Page ID mapping)...
Train data shape: (145063, 551)
Key data shape: (8703780, 2)
Melting the wide-format training data...
Merging training data with key data...

--- Final Merged, Long-Format DataFrame Head ---
                                                Page       Date  Visits   Id
0            2NE1_zh.wikipedia.org_all-access_spider 2015-07-01    18.0  NaN
1             2PM_zh.wikipedia.org_all-access_spider 2015-07-01    11.0  NaN
2              3C_zh.wikipedia.org_all-access_spider 2015-07-01     1.0  NaN
3         4minute_zh.wikipedia.org_all-access_spider 2015-07-01    35.0  NaN
4  52_Hz_I_Love_You_zh.wikipedia.org_all-access_s... 2015-07-01     NaN  NaN

--- DataFrame Information ---
<class 'pandas.core.frame.DataFrame'>

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()



Starting Stage 2 Data Load...
Loading data for Stage 2...
Loading /kaggle/input/web-traffic-time-series-forecasting/train_2.csv.zip (Web Traffic Views)...
Loading /kaggle/input/web-traffic-time-series-forecasting/key_2.csv.zip (Page ID mapping)...
Train data shape: (145063, 804)
Key data shape: (8993906, 2)
Melting the wide-format training data...
Merging training data with key data...


### Check missing value 

In [ ]:
check_missing_values(final_merged_data)

### Check duplicate vakue

In [ ]:
check_duplicates(final_merged_data)

### Validate the Timestamp Sequence:

In [ ]:
validate_timestamp_sequence(final_merged_data) 